In [46]:
import sys
sys.path.append('..')

from utils.spark_session import get_spark_session
from utils.parquet_io import read_parquet, write_parquet
from pyspark.sql.functions import col, dayofweek, hour, when, round, to_date, count, sum, avg


# Initialize Spark session
spark = get_spark_session()
print('session created')

session created


In [47]:
# Read parquet files
taxi01_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\04_join_data\\taxi01")
taxi02_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\04_join_data\\taxi02")

In [48]:
# Total trips per day

taxi01_trips_per_day = taxi01_df.groupBy(
    to_date("tpep_pickup_datetime").alias("pickup_date"),"is_weekend"
).agg(
    count("*").alias("total_trips")
).orderBy("pickup_date")

taxi02_trips_per_day = taxi02_df.groupBy(
    to_date("tpep_pickup_datetime").alias("pickup_date"),"is_weekend"
).agg(
    count("*").alias("total_trips")
).orderBy("pickup_date")


In [49]:
taxi02_trips_per_day.show(4)

+-----------+----------+-----------+
|pickup_date|is_weekend|total_trips|
+-----------+----------+-----------+
| 2017-02-01|         0|     316261|
| 2017-02-02|         0|     349384|
| 2017-02-03|         0|     370328|
| 2017-02-04|         1|     366545|
+-----------+----------+-----------+
only showing top 4 rows


In [50]:
# Revenue per day

taxi01_revenue_per_day = taxi01_df.groupBy(
    to_date("tpep_pickup_datetime").alias("trip_date"),"is_weekend"
).agg(
    round(sum("total_amount"), 2).alias("daily_revenue")
).orderBy("trip_date")

taxi02_revenue_per_day = taxi02_df.groupBy(
    to_date("tpep_pickup_datetime").alias("trip_date"),"is_weekend"
).agg(
    round(sum("total_amount"), 2).alias("daily_revenue")
).orderBy("trip_date")



In [51]:
taxi02_revenue_per_day.show(4)

+----------+----------+-------------+
| trip_date|is_weekend|daily_revenue|
+----------+----------+-------------+
|2017-02-01|         0|    5029099.9|
|2017-02-02|         0|   5480921.07|
|2017-02-03|         0|   5670088.66|
|2017-02-04|         1|   5198017.41|
+----------+----------+-------------+
only showing top 4 rows


In [52]:
# Top pickup zones

taxi01_top_pickup_zones = taxi01_df.groupBy("PULocationID", col("PU_zone").alias("zone_name")).agg(
    count("*").alias("total_trips")
).orderBy(col("total_trips").desc())

taxi02_top_pickup_zones = taxi02_df.groupBy("PULocationID", col("PU_zone").alias("zone_name")).agg(
    count("*").alias("total_trips")
).orderBy(col("total_trips").desc())


In [53]:
taxi02_top_pickup_zones.show()

+------------+--------------------+-----------+
|PULocationID|           zone_name|total_trips|
+------------+--------------------+-----------+
|         161|      Midtown Center|     347745|
|         237|Upper East Side S...|     342670|
|         186|Penn Station/Madi...|     324060|
|         236|Upper East Side N...|     321476|
|         234|            Union Sq|     314299|
|         230|Times Sq/Theatre ...|     312184|
|         162|        Midtown East|     308966|
|         170|         Murray Hill|     291950|
|          79|        East Village|     291335|
|          48|        Clinton East|     283436|
|         142| Lincoln Square East|     257654|
|         163|       Midtown North|     241560|
|         239|Upper West Side S...|     229793|
|         164|       Midtown South|     222292|
|         107|            Gramercy|     220671|
|          68|        East Chelsea|     218682|
|         249|        West Village|     208796|
|         141|     Lenox Hill West|     

In [54]:
# Top dropoff zones

taxi01_top_dropoff_zones = taxi01_df.groupBy("DOLocationID", col("DO_zone").alias("zone_name")).agg(
    count("*").alias("total_trips")
).orderBy(col("total_trips").desc())

taxi02_top_dropoff_zones = taxi02_df.groupBy("DOLocationID", col("DO_zone").alias("zone_name")).agg(
    count("*").alias("total_trips")
).orderBy(col("total_trips").desc())


In [55]:
# Avarage distance per hour

taxi01_avg_distance = taxi01_df.groupBy(
    hour("tpep_pickup_datetime").alias("hour_of_day"),"tpep_pickup_datetime"
).agg(
    round(avg("trip_distance"), 2).alias("avg_trip_distance")
).orderBy("hour_of_day")

taxi02_avg_distance = taxi02_df.groupBy(
    hour("tpep_pickup_datetime").alias("hour_of_day"),"tpep_pickup_datetime"
).agg(
    round(avg("trip_distance"), 2).alias("avg_trip_distance")
).orderBy("hour_of_day")


In [57]:
# Save Aggregated data
write_parquet(taxi01_trips_per_day, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\05_aggregate_data\\trips_per_day")

write_parquet(taxi01_revenue_per_day, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\05_aggregate_data\\revenue_per_day")

write_parquet(taxi01_top_pickup_zones, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\05_aggregate_data\\top_pickup_zones")

write_parquet(taxi01_top_dropoff_zones, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\05_aggregate_data\\top_dropoff_zones")

write_parquet(taxi01_avg_distance, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\05_aggregate_data\\avg_distance")

print("Aggregated data saved successfully!")

Aggregated data saved successfully!


In [58]:
spark.stop()